# Train one Tiny Timbre experiment on a Colab GPU

Runs `tiny_timbre.train` from the repo's uv environment, exactly as on any other machine.
Colab only supplies the GPU and Drive.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom.

- Checkpoints and results are saved to Drive, so if Colab disconnects, run all cells again with the same `CONFIG`: training resumes from the last finished epoch.
- One experiment per session. Pick it in the last cell.

In [ ]:
# 1. Drive (spectrograms, checkpoints, results live here)
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/dl-final"   # the team's shared folder (add a shortcut to My Drive if it's missing)

In [ ]:
# 2. Get the latest code
!test -d /content/tiny-timbre || git clone -q https://github.com/mediabymadelyn/tiny-timbre.git /content/tiny-timbre
%cd /content/tiny-timbre
!git checkout -q main && git pull -q
!git log --oneline -1

In [ ]:
# 3. Install the locked environment with uv (also downloads the right Python), then check the GPU
!pip install -q uv
!uv sync -q
!uv run python -c "import torch; print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
# 4. Data onto Colab's local disk (much faster to read than Drive). Skips anything already copied.
!mkdir -p data
!test -f data/mels_medium/logmel_f16.npy || cp -r "$DRIVE/data/mels_medium" data/    # ~8 GB, a few minutes
!test -f data/fma_metadata/tracks.csv || ( (test -f "$DRIVE/data/fma_metadata.zip" && cp "$DRIVE/data/fma_metadata.zip" /content/) || curl -sSo /content/fma_metadata.zip https://os.unil.cloud.switch.ch/fma/fma_metadata.zip ) && unzip -qn /content/fma_metadata.zip -d data/
!ls data data/mels_medium

In [ ]:
# 5. Save checkpoints and results straight to Drive, so a disconnect loses nothing
!mkdir -p "$DRIVE/checkpoints" "$DRIVE/results"
!ln -sfn "$DRIVE/checkpoints" checkpoints
!ln -sfn "$DRIVE/results" data/results
!ls -l checkpoints data/results

In [ ]:
# 6. Weights & Biases login (paste your API key from https://wandb.ai/authorize; it isn't shown or saved)
import os, getpass
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = getpass.getpass("W&B API key: ")

## Pick the experiment and train

Start with `configs/sanity.yaml` (1,000 clips, 3 epochs): the training loss should go down.
Then one real run per session:

| Config | Experiment |
|---|---|
| `configs/e0_cnn_flat.yaml` | E0, and E1 flat seed 0 |
| `configs/e1_flat_s1.yaml`, `configs/e1_flat_s2.yaml` | E1 flat, seeds 1 and 2 |
| `configs/e1_gcn_s0.yaml` | E1 GCN seed 0 (also E2 and E3's comparison run) |
| `configs/e1_gcn_s1.yaml`, `configs/e1_gcn_s2.yaml` | E1 GCN, seeds 1 and 2 |
| `configs/e2_gcn_cooc_s0.yaml` | E2: extra co-occurrence edges |
| `configs/e3_gcn_1layer_s0.yaml`, `configs/e3_gcn_3layer_s0.yaml` | E3: GCN depth |

Each run prints one line per epoch and, at the end, its test-split scores. Live charts: wandb.ai, project `tiny-timbre`.

In [ ]:
CONFIG = "configs/sanity.yaml"
!uv run python -m tiny_timbre.train --config {CONFIG}